# 📄 MarkItDown Converter v2
**Конвертация документов (PDF, DOCX, PPTX, XLSX) в Markdown**

- Источник: `C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_in`
- Результат: `C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_out`
- LLM: **LM Studio** (Qwen2.5 VL 7B) на `http://localhost:1234/v1`
- PDF-скан: автодетект → **EasyOCR** (текст) + **Qwen VL** (описание изображений)
- Если LM Studio недоступен — конвертация продолжается без описания изображений

<div style="border: 3px solid #d32f2f; padding: 14px; border-radius: 10px; background-color: #fdecea; font-family: Arial;">
  <h3 style="margin: 0; color: #d32f2f;">⚠️ ВНИМАНИЕ</h3>
  <p style="margin: 8px 0 0 0;">
    Если текст не извлекается по всем текстовым слоям —<br>
    запусти <b>ячейку №7 (только LLM ) если есть тектовый слой то ячейку № 7А</b>.
  </p>
</div>

---
## 📦 Ячейка 1 — Установка зависимостей
Устанавливаем все нужные пакеты:
- `markitdown[all]` — конвертер документов (DOCX, PPTX, XLSX, текстовые PDF)
- `openai` — клиент для LM Studio (OpenAI-совместимый API)
- `easyocr` — OCR-движок для распознавания текста со сканов (поддерживает русский)
- `pymupdf` — работа с PDF: извлечение текста, рендер страниц в изображения
- `pillow` — работа с изображениями (нужна EasyOCR и для передачи картинок в Qwen VL)

%pip install markitdown[all] openai easyocr pymupdf pillow --quiet
print('✅ Зависимости установлены')

---
## ⚙️ Ячейка 2 — Настройки путей и параметров
Все пути и параметры собраны в одном месте — менять только здесь.

In [1]:
from pathlib import Path

# ── Пути к папкам ──────────────────────────────────────────────
DIR_IN  = Path(r'C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_in')
DIR_OUT = Path(r'C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_out')

# ── Поддерживаемые форматы ─────────────────────────────────────
SUPPORTED_EXTENSIONS = {'.pdf', '.docx', '.pptx', '.xlsx'}

# ── Параметры LM Studio ────────────────────────────────────────
LM_STUDIO_BASE_URL = 'http://localhost:1234/v1'
LM_STUDIO_API_KEY  = 'lm-studio'       # LM Studio принимает любую строку как ключ
LM_STUDIO_MODEL = 'qwen/qwen2.5-vl-7b' # Имя модели как отображается в LM Studio

# ── Параметры детектора PDF-скана ──────────────────────────────
# Если среднее число символов на страницу ниже этого порога — PDF считается сканом.
# Настоящий текстовый PDF даёт обычно 500-3000 символов на страницу.
PDF_SCAN_THRESHOLD = 50

# ── DPI для рендера страниц PDF в изображения ──────────────────
# 150 DPI — достаточно для OCR и быстро; 300 DPI — лучше качество, медленнее
PDF_RENDER_DPI = 150

# ── Создаём папку вывода если её нет ──────────────────────────
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f'📂 Источник      : {DIR_IN}')
print(f'📂 Результат     : {DIR_OUT}')
print(f'📋 Форматы       : {", ".join(sorted(SUPPORTED_EXTENSIONS))}')
print(f'🔍 Порог скана   : {PDF_SCAN_THRESHOLD} симв/страница')
print(f'🖼️  Рендер PDF    : {PDF_RENDER_DPI} DPI')

📂 Источник      : C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_in
📂 Результат     : C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_out
📋 Форматы       : .docx, .pdf, .pptx, .xlsx
🔍 Порог скана   : 50 симв/страница
🖼️  Рендер PDF    : 150 DPI


---
## 🛑 Ячейка 3 — Проверка папки docs_out
Защита от случайной перезаписи: если в docs_out уже есть файлы — останавливаемся.
`raise SystemExit` при Run All прерывает цепочку — ячейки 4-9 не запустятся.

In [3]:
# Собираем все файлы в папке docs_out рекурсивно
existing_files = [f for f in DIR_OUT.rglob('*') if f.is_file()]

if existing_files:
    print('🛑 СТОП — папка docs_out не пустая!')
    print(f'   Найдено файлов: {len(existing_files)}')
    print()
    for f in sorted(existing_files):
        rel = f.relative_to(DIR_OUT)
        size_kb = f.stat().st_size / 1024
        print(f'   • {rel}  ({size_kb:.1f} KB)')
    print()
    print('   Очистите папку вручную или переименуйте её, затем запустите ячейку снова.')
    raise SystemExit('Выполнение прервано: docs_out содержит файлы.')
else:
    print('✅ Папка docs_out пустая — можно продолжать')

✅ Папка docs_out пустая — можно продолжать


---
## 🔌 Ячейка 4 — Проверка подключения к LM Studio
Qwen2.5 VL нужен для двух задач:
1. Описание изображений в PPTX-файлах
2. Описание диаграмм и схем на страницах PDF-сканов (новое)

Если LM Studio недоступен — EasyOCR всё равно распознает текст, просто без описания картинок.

In [4]:
import openai  # openai — клиент для OpenAI-совместимых API, включая LM Studio

llm_client = None

try:
    client_candidate = openai.OpenAI(
        base_url=LM_STUDIO_BASE_URL,
        api_key=LM_STUDIO_API_KEY,
    )
    # Тестовый запрос — если LM Studio не запущен, придёт ConnectionError
    models = client_candidate.models.list()
    available_models = [m.id for m in models.data]
    llm_client = client_candidate
    print('✅ LM Studio доступен!')
    print(f'   Загруженные модели: {available_models}')
    print(f'   Будет использована: {LM_STUDIO_MODEL}')
    print('   📸 Изображения в PPTX и диаграммы в PDF-сканах будут описаны через Qwen VL')
except Exception as e:
    print('⚠️  LM Studio недоступен — конвертация без описания изображений')
    print(f'   Причина: {e}')
    print('   ℹ️  Текст из PDF-сканов EasyOCR всё равно распознает')

✅ LM Studio доступен!
   Загруженные модели: ['qwen2.5-14b-instruct-1m', 'qwen2.5-vl-7b-instruct', 'text-embedding-bge-m3', 'qwen/qwen2.5-vl-7b', 'text-embedding-nomic-embed-text-v1.5']
   Будет использована: qwen/qwen2.5-vl-7b
   📸 Изображения в PPTX и диаграммы в PDF-сканах будут описаны через Qwen VL


---
## 🧠 Ячейка 5 — Вспомогательные функции
Весь вспомогательный инструментарий для работы с PDF-сканами:
- `is_scanned_pdf()` — детектор: текстовый PDF или скан?
- `render_page_to_pil()` — рендер одной страницы PDF в PNG-изображение
- `describe_image_with_llm()` — отправляем изображение в Qwen VL, получаем описание
- `ocr_pdf()` — полный пайплайн: OCR текста + описание картинок, склейка в Markdown

In [5]:
import fitz        # fitz = pymupdf — рендер страниц PDF в изображения
import easyocr     # easyocr — OCR fallback если LM Studio недоступен
import base64      # base64 — кодируем PNG для передачи в API LM Studio
import io          # io — работа с байтами в памяти без записи на диск
import numpy as np # numpy — конвертация PIL в массив для EasyOCR
from PIL import Image

# Подавляем предупреждения pymupdf о нестандартных цветах в PDF
fitz.TOOLS.mupdf_display_errors(False)

# ── Инициализация EasyOCR (только как fallback) ────────────────
# Первый запуск скачает языковые модели ~200 MB — это нормально
print('⏳ Инициализация EasyOCR (fallback если LM Studio недоступен)...')
ocr_reader = easyocr.Reader(['ru', 'en'], gpu=False)
print('✅ EasyOCR готов')
print()


def render_page_to_pil(page, dpi=PDF_RENDER_DPI):
    # zoom = dpi/72 — базовое разрешение PDF = 72 DPI
    zoom = dpi / 72
    pix  = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    return Image.open(io.BytesIO(pix.tobytes('png')))


def describe_page_with_llm(pil_image, page_num):
    # Отправляем страницу целиком в Qwen VL — он видит и текст и графику
    buf = io.BytesIO()
    pil_image.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    try:
        resp = llm_client.chat.completions.create(
            model=LM_STUDIO_MODEL,
            messages=[{'role': 'user', 'content': [
                {'type': 'image_url',
                 'image_url': {'url': f'data:image/png;base64,{b64}'}},
                {'type': 'text', 'text': (
    'Проанализируй страницу и опиши её полностью на русском языке. '
    'Структура layout: опиши расположение элементов — сколько колонок, '
    'строк, блоков/прямоугольников. Для каждого блока опиши его позицию '
    '(верхний левый, верхний центр и т.д.) и всё содержимое внутри. '
    'Например: "Верхний ряд, левый блок: заголовок Методология расчётов, '
    'далее список: Методология РБ и А3, Схемы расчётов...". '
    'Графики и диаграммы: опиши тип (столбчатый, круговой, линейный), '
    'все подписи осей, легенду, конкретные значения и выводы. '
    'Например: "Горизонтальная гистограмма: штрафы ГИБДД — 12%, ЖКХ — 3%". '
    'Таблицы: опиши заголовки столбцов и все строки с данными. '
    'Весь текст выписывай дословно. Никаких вводных фраз, сразу к делу.'
)}
            ]}],
            max_tokens=1500,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f'[Ошибка Qwen VL стр.{page_num}: {e}]'


def ocr_pdf(pdf_path, progress_prefix=''):
    doc = fitz.open(pdf_path)
    md_parts = []

    for page_num, page in enumerate(doc, 1):
        print(f'  {progress_prefix} стр.{page_num}/{len(doc)}', end=' ')
        pil_img = render_page_to_pil(page)

        if llm_client is not None:
            # ── Основной маршрут: Qwen VL видит страницу целиком ──
            # Читает текст + описывает диаграммы/таблицы/схемы
            print('→ Qwen VL...', end=' ')
            page_text = describe_page_with_llm(pil_img, page_num)
            print('✓')
        else:
            # ── Fallback: только EasyOCR, только текст ─────────────
            # Используется если LM Studio не запущен
            print('→ EasyOCR (fallback)...', end=' ')
            ocr_lines = ocr_reader.readtext(np.array(pil_img), detail=0)
            page_text = '\n'.join(ocr_lines)
            print(f'{len(ocr_lines)} строк')

        md_parts.append(f'## Страница {page_num}\n\n{page_text}')

    doc.close()
    # Горизонтальная линия как разделитель между страницами
    return '\n\n---\n\n'.join(md_parts)


print('✅ Функции готовы:')
print('   render_page_to_pil()      — рендер страницы PDF в изображение')
print('   describe_page_with_llm()  — Qwen VL читает текст + описывает графику')
print('   ocr_pdf()                 — основной пайплайн для всех PDF')

Using CPU. Note: This module is much faster with a GPU.


⏳ Инициализация EasyOCR (fallback если LM Studio недоступен)...
✅ EasyOCR готов

✅ Функции готовы:
   render_page_to_pil()      — рендер страницы PDF в изображение
   describe_page_with_llm()  — Qwen VL читает текст + описывает графику
   ocr_pdf()                 — основной пайплайн для всех PDF


---
## 🔍 Ячейка 6 — Сканирование папки источника
Находим все файлы поддерживаемых форматов в `docs_in`.
Для PDF сразу определяем тип: текстовый или скан — чтобы знать заранее что нас ждёт.

In [6]:
if not DIR_IN.exists():
    raise FileNotFoundError(f'❌ Папка источника не найдена: {DIR_IN}')

# rglob('*') — рекурсивно обходит все вложенные папки
all_files = [
    f for f in DIR_IN.rglob('*')
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
]
all_files.sort()

if not all_files:
    print(f'⚠️  Файлов для конвертации не найдено в {DIR_IN}')
else:
    print(f'📋 Найдено файлов: {len(all_files)}')
    print()
    for i, f in enumerate(all_files, 1):
        rel     = f.relative_to(DIR_IN)
        size_kb = f.stat().st_size / 1024
        # Для PDF определяем тип сразу при сканировании — удобно знать заранее
        if f.suffix.lower() == '.pdf':
            scanned = is_scanned_pdf(str(f))
            pdf_tag = '🖼️ СКАН ' if scanned else '📝 ТЕКСТ'
            print(f'  {i:>3}. [PDF {pdf_tag}] {rel}  ({size_kb:.1f} KB)')
        else:
            print(f'  {i:>3}. [{f.suffix.upper()[1:]:9}] {rel}  ({size_kb:.1f} KB)')

📋 Найдено файлов: 13



NameError: name 'is_scanned_pdf' is not defined

## Ячейка 7а — Конвертация текстовых PDF (текст + Qwen VL для изображений):

import time
from markitdown import MarkItDown

# ── Инициализация MarkItDown ───────────────────────────────────
if llm_client is not None:
    md_converter = MarkItDown(
        llm_client=llm_client,
        llm_model=LM_STUDIO_MODEL,
        llm_prompt=(
            'Опиши содержимое изображения подробно на русском языке. '
            'Если это диаграмма или график — опиши конкретные цифры и данные. '
            'Если это схема — опиши элементы и связи между ними. '
            'Никаких вводных фраз, сразу к делу.'
        )
    )
    print('🤖 MarkItDown С LLM')
else:
    md_converter = MarkItDown()
    print('📄 MarkItDown БЕЗ LLM')

print()


def extract_images_from_page(page):
    """
    Извлекает все растровые изображения со страницы PDF.
    page.get_images() — возвращает список (xref, ...) встроенных изображений.
    xref — внутренний номер объекта в PDF, по нему извлекаем байты.
    Возвращает список PIL-изображений.
    """
    images = []
    for img_info in page.get_images(full=True):
        xref = img_info[0]  # xref — идентификатор изображения внутри PDF
        try:
            base_img  = page.parent.extract_image(xref)  # Извлекаем байты
            img_bytes = base_img['image']
            pil_img   = Image.open(io.BytesIO(img_bytes))
            # Пропускаем совсем маленькие изображения — это иконки/декор
            if pil_img.width > 100 and pil_img.height > 100:
                images.append(pil_img)
        except Exception:
            continue
    return images


def convert_text_pdf(pdf_path, progress_prefix=''):
    """
    Пайплайн для текстового PDF:
    1. pymupdf извлекает текстовый слой каждой страницы
    2. pymupdf находит встроенные изображения на странице
    3. Каждое изображение отправляется в Qwen VL для описания
    4. Текст + описания изображений склеиваются в Markdown
    """
    doc = fitz.open(pdf_path)
    md_parts = []

    for page_num, page in enumerate(doc, 1):
        print(f'  {progress_prefix} стр.{page_num}/{len(doc)}', end=' ')

        # Шаг 1: извлекаем текстовый слой через pymupdf
        # get_text('text') — plain text, сохраняет порядок чтения
        page_text = page.get_text('text').strip()

        # Шаг 2: ищем изображения на странице
        page_images = extract_images_from_page(page)
        print(f'текст: {len(page_text)} симв | картинок: {len(page_images)}', end=' ')

        # Шаг 3: описываем каждое изображение через Qwen VL
        image_descriptions = []
        if page_images and llm_client is not None:
            print('→ Qwen VL...', end=' ')
            for i, pil_img in enumerate(page_images, 1):
                desc = describe_page_with_llm(pil_img, page_num)
                if desc:
                    image_descriptions.append(f'**Изображение {i}:** {desc}')
            print('✓')
        else:
            print()

        # Шаг 4: склеиваем текст и описания изображений
        parts = []
        if page_text:
            parts.append(page_text)
        if image_descriptions:
            parts.append('\n'.join(image_descriptions))

        md_parts.append(f'## Страница {page_num}\n\n' + '\n\n'.join(parts))

    doc.close()
    return '\n\n---\n\n'.join(md_parts)


# ── Основной цикл — только PDF файлы ──────────────────────────
results_text_pdf = {'ok': [], 'error': []}
pdf_files = [f for f in all_files if f.suffix.lower() == '.pdf']

if not pdf_files:
    print('⚠️  PDF файлов не найдено в списке')
else:
    print(f'📋 PDF файлов для обработки: {len(pdf_files)}')
    print()
    start = time.time()

    for idx, src_path in enumerate(pdf_files, 1):
        rel_path = src_path.relative_to(DIR_IN)
        progress = f'[{idx}/{len(pdf_files)}]'
        out_path = DIR_OUT / rel_path.with_suffix('.md')
        out_path.parent.mkdir(parents=True, exist_ok=True)

        print(f'{progress} ⏳ {rel_path}')

        try:
            t0 = time.time()
            markdown_text = convert_text_pdf(str(src_path), progress)
            elapsed = time.time() - t0

            header = (
                f'<!-- Источник: {src_path.name} | Метод: TextPDF+Qwen | '
                f'Конвертирован: {time.strftime("%Y-%m-%d %H:%M")} -->\n\n'
            )
            out_path.write_text(header + markdown_text, encoding='utf-8')

            lines = markdown_text.count('\n')
            chars = len(markdown_text)
            print(f'{progress} ✅ {elapsed:.1f}с | {lines} строк | {chars:,} симв')
            results_text_pdf['ok'].append(src_path)

        except Exception as e:
            print(f'{progress} ❌ ОШИБКА: {e}')
            results_text_pdf['error'].append((src_path, str(e)))

        print()

    print('=' * 60)
    print(f'🏁 ЗАВЕРШЕНО за {time.time() - start:.1f}с')
    print(f'   ✅ Успешно : {len(results_text_pdf["ok"])} файлов')
    print(f'   ❌ Ошибки  : {len(results_text_pdf["error"])} файлов')
    print(f'\n📂 Результаты сохранены в: {DIR_OUT}')

---
## 🚀 Ячейка 7 — Конвертация файлов только LLM
Основной цикл с маршрутизацией по типу файла:
- **DOCX / PPTX / XLSX** → MarkItDown (+ Qwen VL для изображений в PPTX)
- **PDF текстовый** → MarkItDown
- **PDF скан** → EasyOCR + Qwen VL → единый `.md` файл

In [ ]:
import time
from markitdown import MarkItDown

if llm_client is not None:
    md_converter = MarkItDown(
        llm_client=llm_client,
        llm_model=LM_STUDIO_MODEL,
        llm_prompt=(
            'Опиши содержимое изображения подробно на русском языке. '
            'Если это диаграмма или график — опиши её структуру, данные и ключевые выводы. '
            'Если это схема — опиши элементы и связи между ними.'
        )
    )
    print('🤖 MarkItDown С LLM (изображения в PPTX будут описаны)')
else:
    md_converter = MarkItDown()
    print('📄 MarkItDown БЕЗ LLM')

print()

results     = {'ok': [], 'error': [], 'ocr': []}
total       = len(all_files)
start_total = time.time()

for idx, src_path in enumerate(all_files, 1):

    rel_path = src_path.relative_to(DIR_IN)
    progress = f'[{idx}/{total}]'
    ext      = src_path.suffix.lower()
    out_path = DIR_OUT / rel_path.with_suffix('.md')
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print(f'{progress} ⏳ {rel_path}')

    try:
        t0 = time.time()

        if ext == '.pdf':
            # Все PDF без исключения — через OCR-пайплайн
            # Детектор убран: кастомные шрифты давали ложные срабатывания
            print(f'{progress}   🖼️  PDF → OCR (EasyOCR + Qwen VL)')
            markdown_text = ocr_pdf(str(src_path), progress)
            method = 'OCR+Qwen' if llm_client else 'OCR'
            results['ocr'].append(src_path)

        else:
            # DOCX / PPTX / XLSX → MarkItDown (+ Qwen VL для изображений в PPTX)
            markdown_text = md_converter.convert(str(src_path)).text_content
            method = 'MarkItDown'

        elapsed = time.time() - t0

        header = (
            f'<!-- Источник: {src_path.name} | Метод: {method} | '
            f'Конвертирован: {time.strftime("%Y-%m-%d %H:%M")} | '
            f'LLM: {"включён" if llm_client else "выключен"} -->\n\n'
        )
        out_path.write_text(header + markdown_text, encoding='utf-8')

        chars = len(markdown_text)
        lines = markdown_text.count('\n')
        print(f'{progress} ✅ [{method}] {elapsed:.1f}с | {lines} строк | {chars:,} симв')
        results['ok'].append(src_path)

    except Exception as e:
        print(f'{progress} ❌ ОШИБКА: {e}')
        results['error'].append((src_path, str(e)))

    print()

total_elapsed = time.time() - start_total
print('=' * 60)
print(f'🏁 КОНВЕРТАЦИЯ ЗАВЕРШЕНА за {total_elapsed:.1f}с')
print(f'   ✅ Успешно           : {len(results["ok"])} файлов')
print(f'   🖼️  из них PDF        : {len(results["ocr"])} файлов (EasyOCR+Qwen VL)')
print(f'   ❌ Ошибки            : {len(results["error"])} файлов')

if results['error']:
    print()
    print('Файлы с ошибками:')
    for path, err in results['error']:
        print(f'  • {path.name}: {err}')

print(f'\n📂 Результаты сохранены в: {DIR_OUT}')

🤖 MarkItDown С LLM (изображения в PPTX будут описаны)

[1/13] ⏳ A3_Процесс онбординга поставщика - Управление изменениями бизнес-процессов - Confluence.pdf
[1/13]   🖼️  PDF → OCR (EasyOCR + Qwen VL)
  [1/13] стр.1/8 → Qwen VL... ✓
  [1/13] стр.2/8 → Qwen VL... ✓
  [1/13] стр.3/8 → Qwen VL... ✓
  [1/13] стр.4/8 → Qwen VL... ✓
  [1/13] стр.5/8 → Qwen VL... ✓
  [1/13] стр.6/8 → Qwen VL... ✓
  [1/13] стр.7/8 → Qwen VL... ✓
  [1/13] стр.8/8 → Qwen VL... ✓
[1/13] ✅ [OCR+Qwen] 1206.6с | 48 строк | 404 симв

[2/13] ⏳ А3.NOTIFICATION - 5.pdf
[2/13]   🖼️  PDF → OCR (EasyOCR + Qwen VL)
  [2/13] стр.1/12 → Qwen VL... ✓
  [2/13] стр.2/12 → Qwen VL... ✓
  [2/13] стр.3/12 → Qwen VL... ✓
  [2/13] стр.4/12 → Qwen VL... ✓
  [2/13] стр.5/12 → Qwen VL... ✓
  [2/13] стр.6/12 → Qwen VL... ✓
  [2/13] стр.7/12 → Qwen VL... ✓
  [2/13] стр.8/12 → Qwen VL... ✓
  [2/13] стр.9/12 → Qwen VL... ✓
  [2/13] стр.10/12 → Qwen VL... ✓
  [2/13] стр.11/12 → Qwen VL... ✓
  [2/13] стр.12/12 → Qwen VL... ✓
[2/13] ✅ [OCR+Qwen]

---
## 👀 Ячейка 8 — Предпросмотр результата
Показываем первые 500 символов каждого `.md` для быстрой проверки качества.

In [ ]:
PREVIEW_CHARS = 500
output_files  = sorted(DIR_OUT.rglob('*.md'))

if not output_files:
    print('⚠️  Нет файлов для предпросмотра')
else:
    print(f'📄 Предпросмотр {len(output_files)} файлов (первые {PREVIEW_CHARS} символов)\n')
    for md_file in output_files:
        print('─' * 60)
        print(f'📄 {md_file.name}')
        print('─' * 60)
        content = md_file.read_text(encoding='utf-8')
        print(content[:PREVIEW_CHARS])
        if len(content) > PREVIEW_CHARS:
            print(f'\n  ... ещё {len(content) - PREVIEW_CHARS:,} символов ...')
        print()

---
## 📊 Ячейка 9 — Сводная статистика
Таблица: исходный файл → метод конвертации → размер Markdown → количество строк.

In [ ]:
rows = []

for src_path in results['ok']:
    rel      = src_path.relative_to(DIR_IN)
    out_path = DIR_OUT / rel.with_suffix('.md')
    src_kb   = src_path.stat().st_size / 1024
    out_kb   = out_path.stat().st_size / 1024 if out_path.exists() else 0
    lines    = out_path.read_text(encoding='utf-8').count('\n') if out_path.exists() else 0
    # Определяем метод по наличию файла в списке OCR-сканов
    method   = 'EasyOCR+Qwen' if src_path in results['ocr'] else 'MarkItDown'

    rows.append({
        'Файл'        : src_path.name[:33],
        'Метод'       : method,
        'Исходник KB' : round(src_kb, 1),
        'Markdown KB' : round(out_kb, 1),
        'Строк'       : lines,
    })

if rows:
    col_widths = {'Файл': 35, 'Метод': 14, 'Исходник KB': 12, 'Markdown KB': 12, 'Строк': 7}
    header_line = '  '.join(k.ljust(v) for k, v in col_widths.items())
    sep_line    = '  '.join('-' * v for v in col_widths.values())
    print(header_line)
    print(sep_line)
    for row in rows:
        print('  '.join(str(row[k]).ljust(v) for k, v in col_widths.items()))
    print(sep_line)
    total_src = sum(r['Исходник KB'] for r in rows)
    total_out = sum(r['Markdown KB'] for r in rows)
    print(f'  Итого: {total_src:.1f} KB  →  {total_out:.1f} KB Markdown')
else:
    print('Нет данных для статистики')